In [6]:
import xarray as xr
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from scipy.stats import wasserstein_distance, spearmanr,  pearsonr
import matplotlib.pyplot as plt


from skimage.metrics import structural_similarity as ssim

from matplotlib.colors import CenteredNorm
from torchview import draw_graph
from torchmetrics.functional.regression import r2_score as torch_r2_score
from torchmetrics.functional.image import structural_similarity_index_measure

from astral import moon

from openpyxl.drawing.image import Image
import json
import dask

from sklearn.ensemble import RandomForestRegressor

In [7]:
target_res = 0.125
min_lon, min_lat, max_lon, max_lat =[-61.0, -47.5, -60.0, -44.875]
split_year = 2023
split_year_test = 2025
todas = False
test = True
min_time, max_time = pd.to_datetime("2009-01-01"), pd.to_datetime("2025-12-31")

sp = "SQA"

fishing_ds = xr.open_dataset(f"../../data/processed/targets/cpue_{target_res}.nc").sel(FAOspp=sp)
fishing = fishing_ds["cpue_index"]
fishing = fishing.fillna(0)

peso = fishing_ds["PesoTotal"]/1000
peso = np.log1p(peso)
peso = peso.fillna(0)

effort = xr.open_dataset(f"../../data/processed/targets/esfuerzo_{target_res}.nc")
effort = effort["Horas"]/24
effort = np.log1p(effort)
effort = effort.fillna(0)

mask_ds = xr.open_dataset(f"../../data/processed/static/area_pesca_{target_res}.nc")
mask = mask_ds["mask"]
mask = mask.fillna(0)
mask = mask.broadcast_like(fishing)

temp_ds = xr.open_dataset("../../data/processed/dynamic/to_surface.nc")
temp_ds = temp_ds.rename({"to":"TO"})
temp = temp_ds["TO"]
# temp = (temp - temp.mean()) / temp.std()
temp = temp.fillna(0)

temp_bottom_ds = xr.open_dataset("../../data/processed/dynamic/temp_bottom.nc")
temp_bottom_ds = temp_bottom_ds.rename({"to": "TOB"})
temp_bottom = temp_bottom_ds["TOB"]
# temp_bottom = (temp_bottom - temp_bottom.mean()) / temp_bottom.std()
temp_bottom = temp_bottom.fillna(0)

chl_ds = xr.open_dataset("../../data/processed/dynamic/chl.nc")
chl = chl_ds["CHL"]
# chl = (chl - chl.mean()) / chl.std()
chl = chl.fillna(0)

mixed_ds = xr.open_dataset("../../data/processed/dynamic/mixed_layer.nc")
mixed_ds = mixed_ds.rename({"mlotst":"MLOTST"})
mixed = mixed_ds["MLOTST"]
# mixed = (mixed - mixed.mean()) / mixed.std()
mixed = mixed.fillna(0)

depth_ds = xr.open_dataset("../../data/processed/static/depth.nc")
depth_ds = depth_ds.rename({"depth":"PROF"})
depth = depth_ds["PROF"]
# depth = (depth - depth.mean()) / depth.std()
depth = depth.fillna(0)
depth = depth.broadcast_like(temp)

zo_ds = xr.open_dataset("../../data/processed/dynamic/zo_surface.nc")
zo_ds = zo_ds.rename({"zo":"ZO"})
zo = zo_ds["ZO"]
# zo = (zo - zo.mean()) / zo.std()
zo = zo.fillna(0)

so_ds = xr.open_dataset("../../data/processed/dynamic/so_surface.nc")
so_ds = so_ds.rename({"so":"SO"})
so = so_ds["SO"]
# so = (so - so.mean()) / so.std()
so = so.fillna(0)

ugo_ds = xr.open_dataset("../../data/processed/dynamic/ugo_surface.nc")
ugo_ds = ugo_ds.rename({"ugo":"UGO"})
ugo = ugo_ds["UGO"]
# ugo = (ugo - ugo.mean()) / ugo.std()
ugo = ugo.fillna(0)

vgo_ds = xr.open_dataset("../../data/processed/dynamic/vgo_surface.nc")
vgo_ds = vgo_ds.rename({"vgo":"VGO"})
vgo = vgo_ds["VGO"]
# vgo = (vgo - vgo.mean()) / vgo.std()
vgo = vgo.fillna(0)

pp = xr.open_dataset("../../data/processed/dynamic/pp.nc")
pp = pp["PP"]
# pp = (pp - pp.mean()) / pp.std()
pp = pp.fillna(0)

cdm = xr.open_dataset("../../data/processed/dynamic/cdm.nc")
cdm = cdm["CDM"]
# cdm = (cdm - cdm.mean()) / cdm.std()
cdm = cdm.fillna(0)

spm = xr.open_dataset("../../data/processed/dynamic/spm.nc")
spm = spm["SPM"]
# spm = (spm - spm.mean()) / spm.std()    
spm = spm.fillna(0)

zsd = xr.open_dataset("../../data/processed/dynamic/zsd.nc")
zsd = zsd["ZSD"]
# zsd = (zsd - zsd.mean()) / zsd.std()
zsd = zsd.fillna(0)


month = temp["time"].dt.month
month_sin = np.sin(2 * np.pi * month / 12)
month_cos = np.cos(2 * np.pi * month / 12)
month_sin = month_sin.broadcast_like(temp)
month_cos = month_cos.broadcast_like(temp)
month_sin.name = "MSEN"
month_cos.name = "MCOS"
month = month.broadcast_like(temp)

times = pd.DatetimeIndex(temp.time.values)

moon_ = np.array([moon.phase(t) for t in times])
moon_phase = xr.DataArray(moon_, coords={"time": temp.time}, dims=["time"], name="FL")
# moon_phase = (moon_phase - moon_phase.mean()) / moon_phase.std()
moon_phase = moon_phase.fillna(0)
moon_phase = moon_phase.broadcast_like(temp)


year = temp["time"].dt.year
# year = (year - year.mean()) / year.std()
year = year.broadcast_like(temp)
year.name = "Año"

lat = temp["lat"]
lon = temp["lon"]
lat = lat.broadcast_like(temp)
lon = lon.broadcast_like(temp)
lat.name = "LAT"
lon.name = "LON"

temp, temp_bottom, chl, mixed, depth, month_sin, month_cos, lat, lon, zo, so, month, year, ugo, vgo, pp, cdm, spm, zsd, moon_phase = xr.align(
    temp, temp_bottom, chl, mixed, depth, month_sin, month_cos, lat, lon, zo, so, month, year, vgo, ugo, pp, cdm, spm, zsd, moon_phase, join="inner")

fishing, peso, effort, mask = xr.align(fishing, peso, effort, mask, join="inner")


#area de pesca y un poco alrededor
croppedT = lambda da: da.sel(
    lon=slice(min_lon-0.5, max_lon+0.5),
    lat=slice(min_lat-0.5, max_lat+0.5),
    time=slice(min_time, max_time)
)

# area global
cropped = lambda da: da.sel(
    lon=slice(min_lon-0, max_lon+2),
    lat=slice(min_lat-4, max_lat+1),
    time=slice(min_time, max_time)
)

fishing = croppedT(fishing)
effort = croppedT(effort)    
mask = croppedT(mask)
peso = croppedT(peso)

temp = cropped(temp)
temp_bottom = cropped(temp_bottom)
chl = cropped(chl)
mixed = cropped(mixed)
depth = cropped(depth)
zo = cropped(zo)
so = cropped(so)
month = cropped(month)
month_sin = cropped(month_sin)
month_cos = cropped(month_cos)
lat = cropped(lat)
lon = cropped(lon)
year = cropped(year)
ugo = cropped(ugo)
vgo = cropped(vgo)
pp = cropped(pp)
cdm = cropped(cdm)
spm = cropped(spm)
zsd = cropped(zsd)
moon_phase = cropped(moon_phase)

In [8]:
y = fishing
y = y.transpose("time", "lat", "lon")

m = mask 
m = m.transpose("time", "lat", "lon")

vars_ = [temp, temp_bottom,
        so,  zo, 
        ugo, vgo, mixed,
        cdm, spm, chl, pp,
        month_sin, month_cos,
        # year,
        # month,
        # moon_phase,
        # lat, lon,
        # depth,
        ]



vars_names = [v.name for v in vars_]
in_channels = len(vars_)

X = xr.concat(vars_, dim="channel")
X = X.transpose("time", "channel", "lat", "lon")

print(X.shape)


# -------- Data Standardization (Z-score Normalization) --------
X = X.assign_coords(channel=vars_names)
epsilon = 1e-8 

# 1. Isolate the training period ONLY to calculate mean/std (prevents data leakage)
train_slice = X.sel(time=slice(None, f"{split_year-1}-12-31"))
train_mean = train_slice.mean(dim=["time", "lat", "lon"], skipna=True)
train_std  = train_slice.std(dim=["time", "lat", "lon"], skipna=True)

# 2. Set mean=0 and std=1 for cyclical features
for feat in ["month_sin", "month_cos"]:
    if feat in train_mean.coords["channel"].values:
        train_mean.loc[dict(channel=feat)] = 0.0
        train_std.loc[dict(channel=feat)] = 1.0 - epsilon

# 3. Apply the transformation to the ENTIRE dataset
X_scaled = (X - train_mean) / (train_std + epsilon)

# -------- Window Creation --------
# We modify this slightly to also return the exact timestamp of the target (y)
def create_windows_full(X_ds, y_ds, m_ds, window=12):
    X_data = X_ds.values   # (time, channels, H, W)
    y_data = y_ds.values   # (time, H, W)
    m_data = m_ds.values   # (time, H, W)
    time_data = y_ds.time.values # Track the timestamps
    
    X_seq, y_seq, m_seq, time_seq = [], [], [], []

    for i in range(len(X_data) - window):
        X_seq.append(X_data[i:i+window]) # 12 months history
        y_seq.append(y_data[i+window])   # 1 month target
        m_seq.append(m_data[i+window])   # 1 month mask target
        time_seq.append(time_data[i+window]) # The timestamp of the target
        
    return (
        torch.tensor(np.stack(X_seq), dtype=torch.float32),
        torch.tensor(np.stack(y_seq), dtype=torch.float32),
        torch.tensor(np.stack(m_seq), dtype=torch.float32),
        pd.to_datetime(time_seq) # Convert to pandas datetime for easy filtering
    )

window = 12

# 4. Create windows over the entire scaled dataset
X_all, y_all, m_all, t_all = create_windows_full(X_scaled, y, m, window)


# -------- Split into Train/Val/Test --------
# 5. Split the tensors based on the target's year
train_mask = t_all.year < split_year
val_mask = (t_all.year >= split_year) & (t_all.year < split_year_test)
test_mask = t_all.year >= split_year_test

X_train, y_train, m_train = X_all[train_mask], y_all[train_mask], m_all[train_mask]
X_val, y_val, m_val = X_all[val_mask], y_all[val_mask], m_all[val_mask]
X_test, y_test, m_test = X_all[test_mask], y_all[test_mask], m_all[test_mask]

print("Train shape:", X_train.shape)  # Will be N - 12
print("Val shape:", X_val.shape)      # Will retain all 24 months
print("Test shape:", X_test.shape)    # Will retain all 24 months

###### Data Loaders ######
batch_size = 24

train_loader = DataLoader(
    TensorDataset(X_train, y_train, m_train),
    batch_size=batch_size,
    shuffle=False
)

val_loader = DataLoader(
    TensorDataset(X_val, y_val, m_val),
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test, m_test),
    batch_size=batch_size,
    shuffle=False
)


(204, 13, 62, 25)
Train shape: torch.Size([156, 12, 13, 62, 25])
Val shape: torch.Size([24, 12, 13, 62, 25])
Test shape: torch.Size([12, 12, 13, 62, 25])


In [9]:
#Get ml predictions

def prep_ml_data(X, y, m=None):
    """
    Converts 5D input tensors and 3D target tensors into 2D arrays 
    compatible with scikit-learn Random Forest. Handles both Torch tensors and Numpy arrays.
    """
    # Move to CPU and convert to numpy if dealing with PyTorch Tensors
    if isinstance(X, torch.Tensor):
        X = X.detach().cpu().numpy()
    if isinstance(y, torch.Tensor):
        y = y.detach().cpu().numpy()
    if isinstance(m, torch.Tensor):
        m = m.detach().cpu().numpy()

    N = X.shape[0]
    
    # Flatten features: (N, T, C, H_x, W_x) -> (N, T * C * H_x * W_x)
    # For your train data: (72, 12, 11, 94, 48) -> (72, 595584)
    X_rf = X.reshape(N, -1)
    
    # Flatten targets: (N, H_y, W_y) -> (N, H_y * W_y)
    # For your train data: (72, 37, 20) -> (72, 740)
    y_rf = y.reshape(N, -1)
    
    if m is not None:
        m_rf = m.reshape(N, -1)
        return X_rf, y_rf, m_rf
    else:
        return X_rf, y_rf

X_train_ml, y_train_ml, m_train_ml = prep_ml_data(X_train, y_train, m_train)
X_val_ml, y_val_ml, m_val_ml = prep_ml_data(X_val, y_val, m_val)
X_test_ml, y_test_ml, m_test_ml = prep_ml_data(X_test, y_test, m_test)



H_val_test, W_val_test = y_val.shape[1], y_val.shape[2]
H_out_test, W_out_test = y_test.shape[1], y_test.shape[2]

m_true_val = m_val_ml.reshape(-1, H_val_test, W_val_test)
y_true_val = y_val_ml.reshape(-1, H_val_test, W_val_test)

m_true = m_test_ml.reshape(-1, H_out_test, W_out_test)
y_true = y_test_ml.reshape(-1, H_out_test, W_out_test)

mask_flat = m_test_ml.astype(bool)

In [10]:
model = RandomForestRegressor(n_estimators=200, max_features="sqrt", n_jobs=-1, random_state=42)

model.fit(X_train_ml, y_train_ml)

y_pred_flat = model.predict(X_test_ml)

# Make sure mask is boolean
mask_flat = m_test_ml.astype(bool)

# Apply mask in flattened space
y_pred_masked_flat = np.where(mask_flat, y_pred_flat, np.nan)
y_true_masked_flat = np.where(mask_flat, y_test_ml, np.nan)

# Reshape back to spatial maps
all_preds = y_pred_masked_flat.reshape(-1, H_out_test, W_out_test)
all_targets = y_true_masked_flat.reshape(-1, H_out_test, W_out_test)

# Coordinates
lats = fishing["lat"].values
lons = fishing["lon"].values

# Safer: generate exactly as many dates as predictions
times = pd.date_range(
    start="2025-01-01",
    periods=all_preds.shape[0],
    freq="MS"
)

# Build xarray Dataset
ds_test = xr.Dataset(
    {
        "pred": (("time", "lat", "lon"), all_preds),
        "target": (("time", "lat", "lon"), all_targets),
    },
    coords={
        "time": times,
        "lat": lats,
        "lon": lons,
    }
)

# Save to NetCDF
ds_test.to_netcdf(f"./predicted/predicted_RF_{sp}.nc", mode="w")